# Retail Data Cleaning Pipeline

## Overview
This notebook documents the data quality issues found in the raw retail sales dataset and the T-SQL steps taken to clean, standardize, and enrich it for downstream analysis.

## Tech Stack
- **Database:** Microsoft SQL Server
- **Language:** T-SQL
- **Environment:** VS Code SQL Notebooks (MSSQL extension)

## Data Sources
Three raw CSV files are loaded into `RetailDB`:

| File | Table | Description |
|---|---|---|
| `customers.csv` | `retail.customers_raw` | Customer master data |
| `products.csv` | `retail.products_raw` | Product catalog |
| `sales.csv` | `retail.sales_raw` | Transactional sales records |

## Pipeline Stages
1. Setup staging tables
2. Clean customers
3. Clean products
4. Clean sales
5. Merge and enrich
6. Compute metrics
7. Validate results
8. Aggregate to order level

## Setup: create clean staging tables:

In [ ]:
USE RetailDB;
GO

-- Customers clean table
IF OBJECT_ID('retail.customers_clean', 'U') IS NOT NULL
    DROP TABLE retail.customers_clean;

CREATE TABLE retail.customers_clean (
    customer_id   VARCHAR(20)   NOT NULL,
    signup_date   DATE          NULL,
    region        NVARCHAR(100) NULL,
    loyalty_tier  NVARCHAR(50)  NULL
);
GO

-- Products clean table
IF OBJECT_ID('retail.products_clean', 'U') IS NOT NULL
    DROP TABLE retail.products_clean;

CREATE TABLE retail.products_clean (
    product_id     VARCHAR(20)    NOT NULL,
    product_name   NVARCHAR(200)  NOT NULL,
    category       NVARCHAR(100)  NOT NULL,
    base_price     DECIMAL(18,2)  NULL,
    supplier_code  NVARCHAR(50)   NULL
);
GO

-- Sales clean table
IF OBJECT_ID('retail.sales_clean', 'U') IS NOT NULL
    DROP TABLE retail.sales_clean;

CREATE TABLE retail.sales_clean (
    order_id         VARCHAR(20)   NOT NULL,
    customer_id      VARCHAR(20)   NOT NULL,
    product_id       VARCHAR(20)   NOT NULL,
    quantity         INT           NOT NULL,
    unit_price       DECIMAL(18,2) NOT NULL,
    order_date       DATE          NULL,
    delivery_status  NVARCHAR(50)  NULL,
    payment_method   NVARCHAR(100) NULL,
    region           NVARCHAR(100) NULL,
    discount_pct     DECIMAL(5,2)  NULL
);
GO

SELECT 'Staging tables created.' AS status;
GO

## 1. Clean Customers

### Issues Found
- Duplicate `customer_id` records
- Invalid `signup_date` values (`2023-13-45` — month 13, day 45)
- Inconsistent `region` casing and spelling (`tehran`, `TEHRAN`, `Tehran Province`)
- Inconsistent `loyalty_tier` casing (`bronze`, `silver`, `SILVER`)

### Cleaning Strategy
- Deduplicate on `customer_id` using `ROW_NUMBER()`
- Parse dates safely with `TRY_CAST`
- Standardize text with `LOWER(LTRIM(RTRIM(...)))` and `CASE` mapping

In [ ]:
USE RetailDB;
GO

WITH normalized AS (
    SELECT
        customer_id,
        TRY_CAST(signup_date AS DATE) AS signup_date,
        region,
        loyalty_tier,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY (SELECT NULL)
        ) AS rn
    FROM retail.customers_raw
    WHERE customer_id IS NOT NULL
)
INSERT INTO retail.customers_clean (customer_id, signup_date, region, loyalty_tier)
SELECT
    customer_id,
    signup_date,
    CASE LOWER(LTRIM(RTRIM(region)))
        WHEN 'tehran'           THEN 'Tehran'
        WHEN 'tehran province'  THEN 'Tehran'
        WHEN 'mashad'           THEN 'Mashhad'
        WHEN 'mashhad'          THEN 'Mashhad'
        WHEN 'isfahan'          THEN 'Isfahan'
        WHEN 'esfahan'          THEN 'Isfahan'
        WHEN 'khorasan razavi'  THEN 'Khorasan Razavi'
        WHEN 'east azerbaijan'  THEN 'East Azerbaijan'
        WHEN 'alborz'           THEN 'Alborz'
        WHEN 'gilan'            THEN 'Gilan'
        WHEN 'mazandaran'       THEN 'Mazandaran'
        WHEN 'yazd'             THEN 'Yazd'
        WHEN 'kerman'           THEN 'Kerman'
        WHEN 'qom'              THEN 'Qom'
        WHEN 'semnan'           THEN 'Semnan'
        WHEN 'hamedan'          THEN 'Hamedan'
        WHEN 'lorestan'         THEN 'Lorestan'
        WHEN 'ardabil'          THEN 'Ardabil'
        WHEN 'ahvaz'            THEN 'Ahvaz'
        WHEN 'shiraz'           THEN 'Shiraz'
        WHEN 'tabriz'           THEN 'Tabriz'
        ELSE NULL
    END,
    CASE LOWER(LTRIM(RTRIM(loyalty_tier)))
        WHEN 'bronze'   THEN 'Bronze'
        WHEN 'silver'   THEN 'Silver'
        WHEN 'gold'     THEN 'Gold'
        WHEN 'platinum' THEN 'Platinum'
        ELSE NULL
    END
FROM normalized
WHERE rn = 1;
GO

-- Verify
SELECT 
    COUNT(*) AS total_clean_customers,
    SUM(CASE WHEN signup_date IS NULL THEN 1 ELSE 0 END) AS null_dates,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS null_regions,
    SUM(CASE WHEN loyalty_tier IS NULL THEN 1 ELSE 0 END) AS null_tiers
FROM retail.customers_clean;
GO

## 2. Clean Products

### Issues Found
- Duplicate product rows
- `base_price = 0` for one headset (`P080`)
- Missing `supplier_code` for `P083`

### Cleaning Strategy
- Deduplicate on `product_id`
- Convert `base_price` to `DECIMAL(18,2)` with `TRY_CAST`
- Set zero/null `base_price` to `NULL`, then impute with category median
- Fill missing `supplier_code` with `'Unknown'`

In [ ]:
USE RetailDB;
GO

WITH normalized AS (
    SELECT
        product_id,
        product_name,
        category,
        TRY_CAST(base_price AS DECIMAL(18,2)) AS base_price,
        supplier_code,
        ROW_NUMBER() OVER (
            PARTITION BY product_id
            ORDER BY (SELECT NULL)
        ) AS rn
    FROM retail.products_raw
    WHERE product_id IS NOT NULL
)
INSERT INTO retail.products_clean (product_id, product_name, category, base_price, supplier_code)
SELECT
    product_id,
    LTRIM(RTRIM(product_name)),
    LTRIM(RTRIM(category)),
    NULLIF(base_price, 0),   -- turn 0 into NULL for later imputation
    COALESCE(NULLIF(LTRIM(RTRIM(supplier_code)), ''), 'Unknown')
FROM normalized
WHERE rn = 1;
GO

SELECT 'products_clean populated' AS status, COUNT(*) AS total FROM retail.products_clean;
GO

## Impute missing base_price with category median:

In [ ]:
USE RetailDB;
GO

WITH median_by_category AS (
    SELECT DISTINCT
        category,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY base_price)
            OVER (PARTITION BY category) AS median_price
    FROM retail.products_clean
    WHERE base_price IS NOT NULL AND base_price > 0
)
UPDATE pc
SET pc.base_price = mbc.median_price
FROM retail.products_clean AS pc
INNER JOIN median_by_category AS mbc
    ON pc.category = mbc.category
WHERE pc.base_price IS NULL;
GO

-- Verify no nulls remain
SELECT 
    COUNT(*) AS total_products,
    SUM(CASE WHEN base_price IS NULL THEN 1 ELSE 0 END) AS null_prices,
    SUM(CASE WHEN base_price <= 0 THEN 1 ELSE 0 END) AS non_positive_prices
FROM retail.products_clean;
GO

## 3. Clean Sales

### Issues Found
- 4 duplicate rows
- Negative quantities (10 rows)
- Zero quantities (4 rows)
- `unit_price` values of `0`, `1`, `100` (placeholder/corrupt entries)
- Missing `payment_method` (23 rows) and `region` (18 rows)
- Inconsistent `delivery_status`, `payment_method`, and `region` casing

### Cleaning Strategy
- Deduplicate on composite key
- Cast types with `TRY_CAST`
- Standardize text values with `CASE` mappings
- Remove rows where `quantity <= 0`, `unit_price < 1000`, or `discount_pct` out of `[0, 100]`
- Leave `NULL` for missing region/payment — will be filled during merge

In [ ]:
USE RetailDB;
GO

WITH normalized AS (
    SELECT
        order_id,
        customer_id,
        product_id,
        TRY_CAST(quantity     AS INT)           AS quantity,
        TRY_CAST(unit_price   AS DECIMAL(18,2)) AS unit_price,
        TRY_CAST(order_date   AS DATE)          AS order_date,
        delivery_status,
        payment_method,
        region,
        TRY_CAST(discount_pct AS DECIMAL(5,2))  AS discount_pct,
        ROW_NUMBER() OVER (
            PARTITION BY order_id, product_id, TRY_CAST(order_date AS DATE)
            ORDER BY (SELECT NULL)
        ) AS rn
    FROM retail.sales_raw
    WHERE order_id    IS NOT NULL
      AND customer_id IS NOT NULL
      AND product_id  IS NOT NULL
)
INSERT INTO retail.sales_clean (
    order_id, customer_id, product_id, quantity, unit_price, order_date,
    delivery_status, payment_method, region, discount_pct
)
SELECT
    order_id,
    LTRIM(RTRIM(customer_id)),
    LTRIM(RTRIM(product_id)),
    quantity,
    unit_price,
    order_date,
    CASE LOWER(LTRIM(RTRIM(delivery_status)))
        WHEN 'delivered' THEN 'Delivered'
        WHEN 'deliverd'  THEN 'Delivered'
        WHEN 'returned'  THEN 'Returned'
        WHEN 'cancelled' THEN 'Cancelled'
        WHEN 'canceled'  THEN 'Cancelled'
        WHEN 'pending'   THEN 'Pending'
        WHEN 'shipped'   THEN 'Shipped'
        ELSE NULL
    END,
    CASE LOWER(REPLACE(REPLACE(LTRIM(RTRIM(payment_method)), '  ', ' '), '  ', ' '))
        WHEN 'card on delivery' THEN 'Card on Delivery'
        WHEN 'online payment'   THEN 'Online Payment'
        WHEN 'onlinepayment'    THEN 'Online Payment'
        WHEN 'bank transfer'    THEN 'Bank Transfer'
        WHEN 'banktransfer'     THEN 'Bank Transfer'
        WHEN 'digipay'          THEN 'DigiPay'
        ELSE NULL
    END,
    CASE LOWER(LTRIM(RTRIM(region)))
        WHEN 'tehran'           THEN 'Tehran'
        WHEN 'tehran province'  THEN 'Tehran'
        WHEN 'mashad'           THEN 'Mashhad'
        WHEN 'mashhad'          THEN 'Mashhad'
        WHEN 'isfahan'          THEN 'Isfahan'
        WHEN 'esfahan'          THEN 'Isfahan'
        WHEN 'khorasan razavi'  THEN 'Khorasan Razavi'
        WHEN 'east azerbaijan'  THEN 'East Azerbaijan'
        WHEN 'alborz'           THEN 'Alborz'
        WHEN 'gilan'            THEN 'Gilan'
        WHEN 'mazandaran'       THEN 'Mazandaran'
        WHEN 'yazd'             THEN 'Yazd'
        WHEN 'kerman'           THEN 'Kerman'
        WHEN 'qom'              THEN 'Qom'
        WHEN 'semnan'           THEN 'Semnan'
        WHEN 'hamedan'          THEN 'Hamedan'
        WHEN 'lorestan'         THEN 'Lorestan'
        WHEN 'ardabil'          THEN 'Ardabil'
        WHEN 'ahvaz'            THEN 'Ahvaz'
        WHEN 'shiraz'           THEN 'Shiraz'
        WHEN 'tabriz'           THEN 'Tabriz'
        ELSE NULL
    END,
    discount_pct
FROM normalized
WHERE rn = 1
  AND quantity     > 0
  AND unit_price   >= 1000
  AND discount_pct BETWEEN 0 AND 100;
GO

SELECT 
    COUNT(*) AS total_clean_sales,
    SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END) AS bad_quantity,
    SUM(CASE WHEN unit_price < 1000 THEN 1 ELSE 0 END) AS bad_price
FROM retail.sales_clean;
GO

## 4. Merge and Enrich

### Strategy
Join `sales_clean` with `products_clean` and `customers_clean` using `LEFT JOIN` to preserve all sales rows. Add two flag columns (`product_known`, `customer_known`) to track referential integrity misses.

Fill missing `region` in this priority:
1. `sales.region` (from the transaction)
2. `customers.region` (from the customer record)
3. `'Unknown'`

In [ ]:
USE RetailDB;
GO

IF OBJECT_ID('retail.sales_enriched', 'U') IS NOT NULL
    DROP TABLE retail.sales_enriched;
GO

SELECT
    s.order_id,
    s.customer_id,
    s.product_id,
    s.quantity,
    s.unit_price,
    s.order_date,
    s.delivery_status,
    s.payment_method,
    COALESCE(s.region, c.region, 'Unknown') AS region,
    s.discount_pct,

    p.product_name,
    p.category,
    p.base_price,
    p.supplier_code,

    c.signup_date,
    COALESCE(c.loyalty_tier, 'Unknown') AS loyalty_tier,

    CASE WHEN p.product_id  IS NULL THEN 0 ELSE 1 END AS product_known,
    CASE WHEN c.customer_id IS NULL THEN 0 ELSE 1 END AS customer_known
INTO retail.sales_enriched
FROM retail.sales_clean      AS s
LEFT JOIN retail.products_clean  AS p ON s.product_id  = p.product_id
LEFT JOIN retail.customers_clean AS c ON s.customer_id = c.customer_id;
GO

SELECT 
    COUNT(*) AS total_rows,
    SUM(product_known)  AS known_products,
    SUM(customer_known) AS known_customers
FROM retail.sales_enriched;
GO

## 5. Compute Metrics

### Formulas
| Metric | Formula | Notes |
|---|---|---|
| `GrossSales` | `quantity × base_price` | Pre-discount revenue |
| `Sales` | `quantity × unit_price` | Actual revenue (post-discount) |
| `DiscountAmount` | `quantity × base_price × discount_pct / 100` | **Never** use `GrossSales − Sales` — it goes negative |
| `PriceDiffPct` | `(unit_price − ExpectedUnitPrice) / ExpectedUnitPrice × 100` | Flags price inconsistencies |
| `PriceAnomaly` | `ABS(PriceDiffPct) > 5` | Rows with >5% deviation |

### Note on Cost / Profit
The dataset does **not** contain a real cost column. `base_price` is a list price, not cost. Any `Profit` or `ProfitMargin` metric would be a **synthetic proxy** and is therefore omitted from this pipeline.

In [ ]:
USE RetailDB;
GO

ALTER TABLE retail.sales_enriched
    ADD GrossSales      DECIMAL(18,2) NULL,
        Sales           DECIMAL(18,2) NULL,
        DiscountAmount  DECIMAL(18,2) NULL,
        ExpectedUnitPrice DECIMAL(18,2) NULL,
        PriceDiffPct    DECIMAL(10,2) NULL,
        PriceAnomaly    BIT           NULL;
GO

UPDATE retail.sales_enriched
SET
    GrossSales        = quantity * base_price,
    Sales             = quantity * unit_price,
    DiscountAmount    = quantity * base_price * discount_pct / 100.0,
    ExpectedUnitPrice = base_price * (1 - discount_pct / 100.0);
GO

UPDATE retail.sales_enriched
SET PriceDiffPct = 
        CASE 
            WHEN ExpectedUnitPrice = 0 THEN NULL
            ELSE ((unit_price - ExpectedUnitPrice) / ExpectedUnitPrice) * 100
        END;
GO

UPDATE retail.sales_enriched
SET PriceAnomaly = CASE WHEN ABS(PriceDiffPct) > 5 THEN 1 ELSE 0 END;
GO

SELECT TOP 5 order_id, GrossSales, Sales, DiscountAmount, PriceDiffPct, PriceAnomaly
FROM retail.sales_enriched
ORDER BY order_id;
GO

## Reading the Validation Output

The two cells below serve distinct purposes and should be read together.

### Cell A — Validation Summary (Counts)

A quick numeric snapshot that answers one question: **is the pipeline clean?**

Each row in the output is an assertion. A clean pipeline returns `0` for every check that represents a fixable problem. Two checks are expected to return non-zero:

- `Missing payment_method` (23) — retained by design, treated as `'Unknown'`
- `Unknown customer IDs` (11) — retained by design, flagged with `customer_known = 0`
- `Unknown product IDs` (9) — retained by design, flagged with `product_known = 0`

Every other row must return `0`. If it doesn't, the pipeline has a bug that must be fixed before proceeding.

### Cell B — Validation Details (Rows)

Answers the follow-up question: **if not clean, exactly what is wrong?**

This cell returns one row per problematic transaction, grouped by check name. It is a forensic list, auditable row by row, and it exists for three reasons:

1. **Auditability** — a reviewer can verify that the flagged rows are genuinely problematic, not false positives.
2. **Escalation** — when you email the payments or CRM team, you can attach the exact `order_id`s that need investigation.
3. **Debugging** — if a future pipeline change causes a check to go from `0` to `500`, this cell shows exactly which rows broke and how.

### Why Two Cells Instead of One

A single SQL cell can only return one result grid. The summary and the details are different shapes (aggregated vs. row-level), so they must be separate cells.

Together they form a **data quality report**:

- The summary is what you show executives — "we have 43 anomalies out of 4,700 rows."
- The details are what you send to engineers — "here are the 43 `order_id`s, categorized."

Real production pipelines generate both. Showing them in your notebook signals that you understand data quality as an ongoing operational concern, not a one-time cleanup.

### Cell A — Validation Summary (Counts):

In [ ]:
USE RetailDB;
GO

SELECT 'Negative quantity'          AS check_name, COUNT(*) AS failures FROM retail.sales_enriched WHERE quantity <= 0
UNION ALL
SELECT 'unit_price < 1000',         COUNT(*) FROM retail.sales_enriched WHERE unit_price < 1000
UNION ALL
SELECT 'base_price <= 0',           COUNT(*) FROM retail.sales_enriched WHERE base_price <= 0
UNION ALL
SELECT 'Negative DiscountAmount',   COUNT(*) FROM retail.sales_enriched WHERE DiscountAmount < 0
UNION ALL
SELECT 'Missing region',            COUNT(*) FROM retail.sales_enriched WHERE region IS NULL OR region = ''
UNION ALL
SELECT 'Missing payment_method',    COUNT(*) FROM retail.sales_enriched WHERE payment_method IS NULL
UNION ALL
SELECT 'Missing loyalty_tier',      COUNT(*) FROM retail.sales_enriched WHERE loyalty_tier IS NULL
UNION ALL
SELECT 'Unknown product IDs',       COUNT(*) FROM retail.sales_enriched WHERE product_known = 0
UNION ALL
SELECT 'Unknown customer IDs',      COUNT(*) FROM retail.sales_enriched WHERE customer_known = 0;
GO

### Cell B — Validation Details (Rows)

In [ ]:
USE RetailDB;
GO

-- =====================================================
-- Validation Details — every row that failed a check
-- =====================================================
SELECT 
    'Negative quantity' AS check_name,
    order_id, customer_id, product_id,
    CONCAT('quantity = ', quantity) AS issue_detail
FROM retail.sales_enriched
WHERE quantity <= 0

UNION ALL

SELECT 
    'unit_price < 1000',
    order_id, customer_id, product_id,
    CONCAT('unit_price = ', unit_price)
FROM retail.sales_enriched
WHERE unit_price < 1000

UNION ALL

SELECT 
    'base_price <= 0',
    order_id, customer_id, product_id,
    CONCAT('base_price = ', base_price)
FROM retail.sales_enriched
WHERE base_price <= 0

UNION ALL

SELECT 
    'Negative DiscountAmount',
    order_id, customer_id, product_id,
    CONCAT('DiscountAmount = ', DiscountAmount)
FROM retail.sales_enriched
WHERE DiscountAmount < 0

UNION ALL

SELECT 
    'Missing region',
    order_id, customer_id, product_id,
    'region is NULL or empty'
FROM retail.sales_enriched
WHERE region IS NULL OR region = ''

UNION ALL

SELECT 
    'Missing payment_method',
    order_id, customer_id, product_id,
    'payment_method is NULL'
FROM retail.sales_enriched
WHERE payment_method IS NULL

UNION ALL

SELECT 
    'Missing loyalty_tier',
    order_id, customer_id, product_id,
    'loyalty_tier is NULL'
FROM retail.sales_enriched
WHERE loyalty_tier IS NULL

UNION ALL

SELECT 
    'Unknown product ID',
    order_id, customer_id, product_id,
    'product_id not found in products table'
FROM retail.sales_enriched
WHERE product_known = 0

UNION ALL

SELECT 
    'Unknown customer ID',
    order_id, customer_id, product_id,
    'customer_id not found in customers table'
FROM retail.sales_enriched
WHERE customer_known = 0

ORDER BY check_name, order_id;
GO

## Data Quality Findings — Open Questions

Three anomalies remain after cleaning and are being tracked rather than silently fixed. This is intentional: silent fixes hide data problems from stakeholders and undermine trust in the pipeline.

### 1. Missing `payment_method` — 23 rows (0.5% of records)

These transactions have valid amounts, customers, and dates but no payment method recorded. Possible causes:

- Alternate ordering channels (phone, B2B invoice) that don't capture payment method
- Payment processor errors during checkout
- Manual data entry gaps
- Free/promotional transactions with no payment

**Treatment:** Retained with `payment_method = 'Unknown'`. Dropping them would understate revenue. Filling them with a guessed method (e.g., `'Online Payment'`) would fabricate data. `'Unknown'` preserves the row and makes the gap visible in any downstream report.

### 2. Unknown `customer_id` — 11 rows (0.2% of records)

These `customer_id` values appear in `sales` but have no match in the `customers` table. Possible causes:

- Customer record deleted post-sale (GDPR / account closure)
- Customer created in a system not synced to `customers`
- Typo during data entry
- CRM sync lag — sale recorded before customer was created
- Test / dummy data leaked into production

**Treatment:** Retained for revenue analysis, excluded from customer-level analysis via the `customer_known = 0` flag. The sales happened and the revenue is real; only the customer identity is missing.

### 3. Unknown `product_id` — 9 rows (0.2% of records)

These `product_id` values appear in `sales` but have no match in the `products` table. Affected IDs include `P240`, `P243`, `P252`, `P257`, `P258`, `P265`, `P283`, `P286`. Possible causes:

- Product was discontinued and removed from the catalog
- Product exists in a system not synced to `products`
- Typo during data entry
- New product not yet catalogued when the sale occurred
- Test / dummy data leaked into production

**Treatment:** Retained for revenue analysis, excluded from product-level analysis via the `product_known = 0` flag. However, note that these rows have **no `base_price`, no `category`, and no `product_name`**, which means:

- `GrossSales` and `DiscountAmount` are `NULL` for these rows
- Category breakdowns will not include them
- Product ranking tables will not include them

Revenue from these rows can still be counted via `Sales` (which only needs `quantity × unit_price`), but any metric that depends on `base_price` will be incomplete.

### Business Impact

Combined, these anomalies account for **less than 1% of transactions** (43 rows out of ~4,700). They do not materially affect total revenue, time-series trends, or overall distribution shapes. They **do** affect:

- Payment-method distribution (23 rows shift to `'Unknown'`)
- Customer concentration analysis (11 sales are unattributed)
- Product ranking and category breakdowns (9 sales have no category)
- Gross-vs-net discount analysis (9 rows have `NULL` `GrossSales`)

### Recommended Actions (Real-World Context)

In a production environment, these findings would be escalated to:

| Anomaly | Owner to Contact | Question |
|---|---|---|
| Missing `payment_method` | Payments team / checkout engineer | Is this an expected gap or a data pipeline bug? |
| Unknown `customer_id` | CRM team / data platform engineer | Are customer records being deleted or lagged? |
| Unknown `product_id` | Catalog team / merchandising | Are products being removed from the catalog, or is this a sync issue? |

Until those answers arrive, the treatments above are documented as **analyst assumptions** and should be reviewed before any production reporting.

## 7. Order-Level Aggregation

Roll up line-item data to one row per order for customer and order-level analysis.

### Aggregations
- `Sales`, `GrossSales`, `DiscountAmount` → SUM
- `order_date` → MIN
- `delivery_status` → MAX (or LAST, depending on business rule)
- `distinct_products` → COUNT DISTINCT

In [ ]:
USE RetailDB;
GO

IF OBJECT_ID('retail.order_summary', 'U') IS NOT NULL
    DROP TABLE retail.order_summary;
GO

SELECT
    order_id,
    SUM(Sales)                    AS Sales,
    SUM(GrossSales)               AS GrossSales,
    SUM(DiscountAmount)           AS DiscountAmount,
    MIN(order_date)               AS order_date,
    MAX(delivery_status)          AS delivery_status,
    COUNT(DISTINCT product_id)    AS distinct_products,
    COUNT(*)                      AS line_items
INTO retail.order_summary
FROM retail.sales_enriched
GROUP BY order_id;
GO

SELECT TOP 10 * FROM retail.order_summary ORDER BY Sales DESC;
GO

## Pre-Analysis Data Integrity Check

Before handing this dataset to the Python EDA stage, we run one last integrity check. Unlike the earlier validation (which hunted for bad rows), this check confirms the dataset is **structurally sound and ready for analysis**.

### What This Check Verifies

| Check | Why It Matters |
|---|---|
| **Row count** | Confirms the pipeline didn't silently drop or duplicate records |
| **Primary key uniqueness** | Confirms no duplicate `(order_id, product_id)` pairs — critical for correct aggregations |
| **Null key columns** | Confirms `order_id`, `customer_id`, `product_id` are all populated |
| **Date range** | Confirms `order_date` values fall in an expected window |
| **Numeric ranges** | Confirms no lingering negatives or zeros in `quantity`, `unit_price`, `Sales` |
| **Metric sanity** | Confirms `Sales ≤ GrossSales` when `discount_pct > 0` |
| **Referential fill rate** | Confirms most rows have category and loyalty info; the small remainder is documented |

### Expected Result

Every check should return a value that matches the documented expectation. A failure does not necessarily mean the pipeline is broken — it means something needs investigation before analysis begins. The goal is **no surprises** once Python starts consuming the data.

### Downstream Impact

The Python EDA stage assumes:

- `sales_enriched` is the single source of truth for line-item analysis
- `order_summary` is the single source of truth for order-level analysis
- Unknown product and customer rows are flagged, not deleted, so they can be included or excluded per analysis
- `Sales` is always populated (never `NULL`) for every retained row
- `GrossSales` and `DiscountAmount` may be `NULL` for the 9 unknown-product rows

## Silver Layer — Advanced Data Quality Validation

### Extended DQ Checks (BLOCKING / WARNING severity)

In [ ]:
USE RetailDB;
GO

-- ============================================================
-- SILVER DQ EXTENSION — Advanced Validation Checks
-- Appends to the existing integrity check pattern.
-- All checks return (check_name, severity, count, detail).
-- severity: 'BLOCKING' = stop pipeline; 'WARNING' = log and continue
-- ============================================================

WITH

-- CHECK 1: Negative or zero quantity (data corruption / system error)
-- Blocking: a sale with qty ≤ 0 makes revenue math meaningless.
negative_qty AS (
    SELECT
        'Negative or zero quantity'      AS check_name,
        'BLOCKING'                       AS severity,
        COUNT(*)                         AS issue_count,
        'order_id sample: '
            + STRING_AGG(CAST(order_id AS NVARCHAR(20)), ', ')
              WITHIN GROUP (ORDER BY order_id)  AS detail
    FROM (
        SELECT TOP 5 order_id
        FROM retail.sales_enriched
        WHERE quantity <= 0
    ) sub
),

-- CHECK 2: Future-dated orders (order_date > today)
-- Blocking: future dates break LAG/cohort index calculations.
future_orders AS (
    SELECT
        'Future-dated orders'            AS check_name,
        'BLOCKING'                       AS severity,
        COUNT(*)                         AS issue_count,
        'Max order_date: ' + CAST(MAX(order_date) AS VARCHAR(20)) AS detail
    FROM retail.sales_enriched
    WHERE order_date > CAST(GETDATE() AS DATE)
),

-- CHECK 3: Order dates before plausible business start (2020-01-01)
-- Warning: might be a test record or bad ETL.
implausible_old_orders AS (
    SELECT
        'Order date before 2020-01-01'   AS check_name,
        'WARNING'                        AS severity,
        COUNT(*)                         AS issue_count,
        'Min order_date: ' + CAST(MIN(order_date) AS VARCHAR(20)) AS detail
    FROM retail.sales_enriched
    WHERE order_date < '2020-01-01'
),

-- CHECK 4: Sales consistency — Sales should equal quantity × unit_price
-- Tolerance: ±0.02 to allow for rounding in source system.
-- Warning (not blocking) because the column is already derived; flag for audit.
sales_consistency AS (
    SELECT
        'Sales ≠ quantity × unit_price (±0.02 tolerance)' AS check_name,
        'WARNING'                                          AS severity,
        COUNT(*)                                           AS issue_count,
        'Max abs diff: '
            + CAST(MAX(ABS(Sales - CAST(quantity AS DECIMAL(18,2)) * unit_price)) AS VARCHAR(20))
            AS detail
    FROM retail.sales_enriched
    WHERE ABS(Sales - CAST(quantity AS DECIMAL(18,2)) * unit_price) > 0.02
),

-- CHECK 5: GrossSales consistency — GrossSales = quantity × base_price
-- Only meaningful for known products (base_price is NULL otherwise).
gross_sales_consistency AS (
    SELECT
        'GrossSales ≠ quantity × base_price (known products, ±0.02)' AS check_name,
        'WARNING'                                                      AS severity,
        COUNT(*)                                                       AS issue_count,
        'Max abs diff: '
            + CAST(MAX(ABS(GrossSales - CAST(quantity AS DECIMAL(18,2)) * base_price)) AS VARCHAR(20))
            AS detail
    FROM retail.sales_enriched
    WHERE product_known = 1
      AND ABS(GrossSales - CAST(quantity AS DECIMAL(18,2)) * base_price) > 0.02
),

-- CHECK 6: Statistical volume anomaly — orders per day > mean + 3σ
-- Uses window functions to compute daily order counts, then flags outlier days.
-- Warning: spikes might be legitimate (promotions) but worth surfacing.
daily_order_counts AS (
    SELECT
        order_date,
        COUNT(DISTINCT order_id) AS daily_orders
    FROM retail.sales_enriched
    GROUP BY order_date
),
daily_stats AS (
    SELECT
        AVG(CAST(daily_orders AS FLOAT))  AS mean_orders,
        STDEV(CAST(daily_orders AS FLOAT)) AS stdev_orders
    FROM daily_order_counts
),
volume_anomaly AS (
    SELECT
        'Days with order volume > mean + 3σ' AS check_name,
        'WARNING'                             AS severity,
        COUNT(*)                              AS issue_count,
        'Threshold: '
            + CAST(CAST(ds.mean_orders + 3 * ds.stdev_orders AS DECIMAL(10,2)) AS VARCHAR(20))
            + ' orders/day'                   AS detail
    FROM daily_order_counts doc
    CROSS JOIN daily_stats ds
    WHERE doc.daily_orders > ds.mean_orders + 3 * ds.stdev_orders
),

-- CHECK 7: Discount outside declared range (should be 0.0–15.0 after Silver cleaning)
-- Blocking: out-of-range discounts corrupt DiscountAmount in Gold.
discount_range AS (
    SELECT
        'discount_pct outside [0, 15]'   AS check_name,
        'BLOCKING'                        AS severity,
        COUNT(*)                          AS issue_count,
        'Range seen: ['
            + CAST(MIN(discount_pct) AS VARCHAR(10)) + ', '
            + CAST(MAX(discount_pct) AS VARCHAR(10)) + ']' AS detail
    FROM retail.sales_enriched
    WHERE discount_pct < 0 OR discount_pct > 15
),

-- CHECK 8: Orphaned sales — customer_known=0 AND product_known=0
-- Both foreign keys missing: line item has no analytical value.
double_orphan AS (
    SELECT
        'Rows with both customer_known=0 and product_known=0' AS check_name,
        'WARNING'                                              AS severity,
        COUNT(*)                                               AS issue_count,
        CAST(ROUND(COUNT(*) * 100.0
            / NULLIF((SELECT COUNT(*) FROM retail.sales_enriched), 0), 2) AS VARCHAR(10))
            + '% of total rows'                                AS detail
    FROM retail.sales_enriched
    WHERE customer_known = 0 AND product_known = 0
)

SELECT check_name, severity, issue_count, detail FROM negative_qty
UNION ALL SELECT check_name, severity, issue_count, detail FROM future_orders
UNION ALL SELECT check_name, severity, issue_count, detail FROM implausible_old_orders
UNION ALL SELECT check_name, severity, issue_count, detail FROM sales_consistency
UNION ALL SELECT check_name, severity, issue_count, detail FROM gross_sales_consistency
UNION ALL SELECT check_name, severity, issue_count, detail FROM volume_anomaly
UNION ALL SELECT check_name, severity, issue_count, detail FROM discount_range
UNION ALL SELECT check_name, severity, issue_count, detail FROM double_orphan
ORDER BY
    CASE severity WHEN 'BLOCKING' THEN 0 ELSE 1 END,
    issue_count DESC;
GO

### DQ Audit Trail Table

In [ ]:
USE RetailDB;
GO

IF OBJECT_ID('retail.gold_validation_log', 'U') IS NULL
BEGIN
    CREATE TABLE retail.gold_validation_log (
        log_id          INT IDENTITY(1,1) PRIMARY KEY,
        run_timestamp   DATETIME2         DEFAULT SYSDATETIME(),
        run_id          VARCHAR(50)       NOT NULL,   -- passed from pipeline orchestrator
        check_name      NVARCHAR(200)     NOT NULL,
        severity        VARCHAR(20)       NOT NULL,
        issue_count     INT               NOT NULL,
        detail          NVARCHAR(500)     NULL,
        pipeline_action VARCHAR(50)       NOT NULL    -- 'BLOCKED' | 'WARNED' | 'PASSED'
    );
END
GO

### Write DQ Results to Audit Log

Insert the Silver DQ check results into `retail.gold_validation_log` for every pipeline run. Replace `'notebook-run-001'` with a timestamp or CI job ID in a scheduled pipeline.

In [ ]:
USE RetailDB;
GO

-- Insert current Silver DQ results into the audit log.
-- pipeline_action is derived from severity:
--   BLOCKING checks with issue_count > 0 → 'BLOCKED'
--   WARNING  checks with issue_count > 0 → 'WARNED'
--   Any check with issue_count = 0       → 'PASSED'
--
-- Replace 'notebook-run-001' with a dynamic run_id in a scheduled pipeline
-- (e.g. CONVERT(VARCHAR, SYSDATETIME(), 120) for a timestamp-based id).

DECLARE @run_id VARCHAR(50) = 'notebook-run-001';

INSERT INTO retail.gold_validation_log (run_id, check_name, severity, issue_count, detail, pipeline_action)

WITH

negative_qty AS (
    SELECT
        'Negative or zero quantity'      AS check_name,
        'BLOCKING'                       AS severity,
        COUNT(*)                         AS issue_count,
        'order_id sample: '
            + ISNULL(STRING_AGG(CAST(order_id AS NVARCHAR(20)), ', ')
              WITHIN GROUP (ORDER BY order_id), 'none')  AS detail
    FROM (
        SELECT TOP 5 order_id
        FROM retail.sales_enriched
        WHERE quantity <= 0
    ) sub
),
future_orders AS (
    SELECT
        'Future-dated orders'            AS check_name,
        'BLOCKING'                       AS severity,
        COUNT(*)                         AS issue_count,
        'Max order_date: ' + ISNULL(CAST(MAX(order_date) AS VARCHAR(20)), 'n/a') AS detail
    FROM retail.sales_enriched
    WHERE order_date > CAST(GETDATE() AS DATE)
),
implausible_old_orders AS (
    SELECT
        'Order date before 2020-01-01'   AS check_name,
        'WARNING'                        AS severity,
        COUNT(*)                         AS issue_count,
        'Min order_date: ' + ISNULL(CAST(MIN(order_date) AS VARCHAR(20)), 'n/a') AS detail
    FROM retail.sales_enriched
    WHERE order_date < '2020-01-01'
),
sales_consistency AS (
    SELECT
        'Sales ≠ quantity × unit_price (±0.02 tolerance)' AS check_name,
        'WARNING'                                          AS severity,
        COUNT(*)                                           AS issue_count,
        'Max abs diff: '
            + ISNULL(CAST(MAX(ABS(Sales - CAST(quantity AS DECIMAL(18,2)) * unit_price)) AS VARCHAR(20)), '0')
            AS detail
    FROM retail.sales_enriched
    WHERE ABS(Sales - CAST(quantity AS DECIMAL(18,2)) * unit_price) > 0.02
),
gross_sales_consistency AS (
    SELECT
        'GrossSales ≠ quantity × base_price (known products, ±0.02)' AS check_name,
        'WARNING'                                                      AS severity,
        COUNT(*)                                                       AS issue_count,
        'Max abs diff: '
            + ISNULL(CAST(MAX(ABS(GrossSales - CAST(quantity AS DECIMAL(18,2)) * base_price)) AS VARCHAR(20)), '0')
            AS detail
    FROM retail.sales_enriched
    WHERE product_known = 1
      AND ABS(GrossSales - CAST(quantity AS DECIMAL(18,2)) * base_price) > 0.02
),
daily_order_counts AS (
    SELECT order_date, COUNT(DISTINCT order_id) AS daily_orders
    FROM retail.sales_enriched
    GROUP BY order_date
),
daily_stats AS (
    SELECT AVG(CAST(daily_orders AS FLOAT)) AS mean_orders,
           STDEV(CAST(daily_orders AS FLOAT)) AS stdev_orders
    FROM daily_order_counts
),
volume_anomaly AS (
    SELECT
        'Days with order volume > mean + 3σ' AS check_name,
        'WARNING'                             AS severity,
        COUNT(*)                              AS issue_count,
        'Threshold: '
            + CAST(CAST(ds.mean_orders + 3 * ds.stdev_orders AS DECIMAL(10,2)) AS VARCHAR(20))
            + ' orders/day'                   AS detail
    FROM daily_order_counts doc
    CROSS JOIN daily_stats ds
    WHERE doc.daily_orders > ds.mean_orders + 3 * ds.stdev_orders
),
discount_range AS (
    SELECT
        'discount_pct outside [0, 15]'   AS check_name,
        'BLOCKING'                        AS severity,
        COUNT(*)                          AS issue_count,
        'Range seen: ['
            + CAST(MIN(discount_pct) AS VARCHAR(10)) + ', '
            + CAST(MAX(discount_pct) AS VARCHAR(10)) + ']' AS detail
    FROM retail.sales_enriched
    WHERE discount_pct < 0 OR discount_pct > 15
),
double_orphan AS (
    SELECT
        'Rows with both customer_known=0 and product_known=0' AS check_name,
        'WARNING'                                              AS severity,
        COUNT(*)                                               AS issue_count,
        CAST(ROUND(COUNT(*) * 100.0
            / NULLIF((SELECT COUNT(*) FROM retail.sales_enriched), 0), 2) AS VARCHAR(10))
            + '% of total rows'                                AS detail
    FROM retail.sales_enriched
    WHERE customer_known = 0 AND product_known = 0
),

all_checks AS (
    SELECT check_name, severity, issue_count, detail FROM negative_qty
    UNION ALL SELECT check_name, severity, issue_count, detail FROM future_orders
    UNION ALL SELECT check_name, severity, issue_count, detail FROM implausible_old_orders
    UNION ALL SELECT check_name, severity, issue_count, detail FROM sales_consistency
    UNION ALL SELECT check_name, severity, issue_count, detail FROM gross_sales_consistency
    UNION ALL SELECT check_name, severity, issue_count, detail FROM volume_anomaly
    UNION ALL SELECT check_name, severity, issue_count, detail FROM discount_range
    UNION ALL SELECT check_name, severity, issue_count, detail FROM double_orphan
)

SELECT
    @run_id,
    check_name,
    severity,
    issue_count,
    detail,
    CASE
        WHEN severity = 'BLOCKING' AND issue_count > 0 THEN 'BLOCKED'
        WHEN severity = 'WARNING'  AND issue_count > 0 THEN 'WARNED'
        ELSE 'PASSED'
    END AS pipeline_action
FROM all_checks;

-- Confirm rows written
SELECT run_id, check_name, severity, issue_count, pipeline_action
FROM retail.gold_validation_log
WHERE run_id = @run_id
ORDER BY CASE pipeline_action WHEN 'BLOCKED' THEN 0 WHEN 'WARNED' THEN 1 ELSE 2 END;
GO

In [ ]:
USE RetailDB;
GO

-- =====================================================
-- Pre-Analysis Data Integrity Check
-- Purpose: Confirm the dataset is structurally sound
--          before handing off to Python EDA.
-- =====================================================

-- 1. Row counts — should match expectations
SELECT 'Total rows in sales_enriched'          AS check_name, CAST(COUNT(*) AS VARCHAR(50)) AS value FROM retail.sales_enriched
UNION ALL
SELECT 'Distinct orders in sales_enriched',    CAST(COUNT(DISTINCT order_id) AS VARCHAR(50)) FROM retail.sales_enriched
UNION ALL
SELECT 'Total rows in order_summary',          CAST(COUNT(*) AS VARCHAR(50)) FROM retail.order_summary
UNION ALL
SELECT 'Total rows in customers_clean',        CAST(COUNT(*) AS VARCHAR(50)) FROM retail.customers_clean
UNION ALL
SELECT 'Total rows in products_clean',         CAST(COUNT(*) AS VARCHAR(50)) FROM retail.products_clean

UNION ALL
-- 2. Primary key uniqueness — should be unique per (order_id, product_id)
SELECT 'Duplicate (order_id, product_id) pairs',
       CAST(COUNT(*) AS VARCHAR(50))
FROM (
    SELECT order_id, product_id
    FROM retail.sales_enriched
    GROUP BY order_id, product_id
    HAVING COUNT(*) > 1
) AS dup

UNION ALL
-- 3. Null key columns — should be 0
SELECT 'Null order_id',                        CAST(COUNT(*) AS VARCHAR(50)) FROM retail.sales_enriched WHERE order_id IS NULL
UNION ALL
SELECT 'Null customer_id',                     CAST(COUNT(*) AS VARCHAR(50)) FROM retail.sales_enriched WHERE customer_id IS NULL
UNION ALL
SELECT 'Null product_id',                      CAST(COUNT(*) AS VARCHAR(50)) FROM retail.sales_enriched WHERE product_id IS NULL

UNION ALL
-- 4. Date range — sanity check
SELECT 'Min order_date',                       CAST(MIN(order_date) AS VARCHAR(50)) FROM retail.sales_enriched
UNION ALL
SELECT 'Max order_date',                       CAST(MAX(order_date) AS VARCHAR(50)) FROM retail.sales_enriched

UNION ALL
-- 5. Numeric sanity — should all be 0
SELECT 'Quantity <= 0',                        CAST(COUNT(*) AS VARCHAR(50)) FROM retail.sales_enriched WHERE quantity <= 0
UNION ALL
SELECT 'Unit price <= 0',                      CAST(COUNT(*) AS VARCHAR(50)) FROM retail.sales_enriched WHERE unit_price <= 0
UNION ALL
SELECT 'Sales <= 0',                           CAST(COUNT(*) AS VARCHAR(50)) FROM retail.sales_enriched WHERE Sales <= 0
UNION ALL
SELECT 'Null Sales',                           CAST(COUNT(*) AS VARCHAR(50)) FROM retail.sales_enriched WHERE Sales IS NULL

UNION ALL
-- 6. Metric sanity: Sales should never exceed GrossSales when discount > 0
SELECT 'Sales > GrossSales when discount > 0',
       CAST(COUNT(*) AS VARCHAR(50))
FROM retail.sales_enriched
WHERE discount_pct > 0
  AND Sales > GrossSales

UNION ALL
-- 7. Referential fill rate — how many rows have product/customer info
SELECT 'Rows with product info (product_known = 1)',
       CAST(SUM(product_known) AS VARCHAR(50))
FROM retail.sales_enriched

UNION ALL
SELECT 'Rows with customer info (customer_known = 1)',
       CAST(SUM(customer_known) AS VARCHAR(50))
FROM retail.sales_enriched;
GO

## Analytics Layer

This section builds the **ten** analytical tables that feed directly into the downstream Python EDA notebook. All computation happens here in SQL — Python will only load these objects and plot. No aggregation, scoring, or metric calculation occurs on the Python side.

| Table | Grain | Primary Use |
|---|---|---|
| `retail.customer_metrics` | customer_id | RFM segmentation, retention |
| `retail.customer_clv` | customer_id | Customer Lifetime Value, CLV segments |
| `retail.product_metrics` | product_id | Product ranking, category performance, return analysis |
| `retail.supplier_metrics` | supplier_code | Supplier scorecard, return-rate outliers |
| `retail.time_series_monthly` | year-month | Revenue trend, MoM growth, seasonality |
| `retail.cohort_base` | customer_id × order_id | Cohort retention (raw) |
| `retail.cohort_retention_rates` | cohort_month × cohort_index | Cohort retention heatmap (pre-pivoted %) |
| `retail.category_monthly_matrix` | category × year-month | Category × month revenue pivot |
| `retail.payment_risk_profile` | payment_method | Payment risk dashboard |
| `retail.mba_order_product_matrix` | order_id × product_id | Market basket analysis (Python Apriori input) |

### Object 1 — `retail.customer_metrics`

**Grain:** One row per `customer_id` (known customers only, `customer_known = 1`).

**Purpose:** Aggregates purchasing behaviour per customer and computes an RFM (Recency–Frequency–Monetary) score. Recency is measured as the gap in days between a customer's last purchase and the latest order date in the entire dataset, so lower recency = more recent = better. Each R/F/M dimension is binned into quintiles (1–5) using `NTILE(5)`; the three scores are concatenated into a single `rfm_segment` string (e.g. `"543"`).

**Business use:** Customer segmentation, retention analysis, lifetime value ranking, loyalty tier validation.

In [ ]:
USE RetailDB;
GO

-- Idempotency guard
IF OBJECT_ID('retail.customer_metrics', 'U') IS NOT NULL
    DROP TABLE retail.customer_metrics;
GO

-- Step 1: compute the dataset-wide latest order date once
-- so recency_days is consistent across all customers.
WITH dataset_max AS (
    SELECT MAX(order_date) AS max_order_date
    FROM retail.sales_enriched
),

-- Step 2: pull stable customer attributes from customers_clean (single source of truth).
-- This avoids grouping on region/loyalty_tier/signup_date from sales_enriched,
-- which can differ across orders for the same customer and inflate row count.
customer_attrs AS (
    SELECT
        customer_id,
        -- Use MAX() as a tie-breaker in case of any residual duplicates in customers_clean
        MAX(loyalty_tier)  AS loyalty_tier,
        MAX(signup_date)   AS signup_date
    FROM retail.customers_clean
    GROUP BY customer_id
),

-- Step 3: aggregate sales to customer grain — only group by customer_id.
-- region is resolved as the customer's modal (most frequent) shipping region
-- using a ROW_NUMBER trick to avoid a multi-column GROUP BY explosion.
customer_region AS (
    SELECT
        customer_id,
        region,
        COUNT(*) AS region_count,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY COUNT(*) DESC, region ASC   -- DESC count, ASC region breaks ties deterministically
        ) AS rn
    FROM retail.sales_enriched
    WHERE customer_known = 1
    GROUP BY customer_id, region
),

customer_agg AS (
    SELECT
        se.customer_id,
        MIN(se.order_date)                                   AS first_purchase_date,
        MAX(se.order_date)                                   AS last_purchase_date,
        COUNT(DISTINCT se.order_id)                          AS total_orders,
        SUM(COALESCE(se.Sales, 0))                           AS total_revenue,
        SUM(COALESCE(se.quantity, 0))                        AS total_units
    FROM retail.sales_enriched se
    WHERE se.customer_known = 1
    GROUP BY se.customer_id   -- single column: guaranteed one row per customer_id
),

-- Step 4: join stable attributes back in (one row per customer_id guaranteed)
customer_joined AS (
    SELECT
        ca.customer_id,
        cr.region,                    -- modal shipping region
        attrs.loyalty_tier,           -- from customers_clean (stable)
        attrs.signup_date,            -- from customers_clean (stable)
        ca.first_purchase_date,
        ca.last_purchase_date,
        ca.total_orders,
        ca.total_revenue,
        ca.total_units
    FROM customer_agg ca
    INNER JOIN customer_attrs attrs ON ca.customer_id = attrs.customer_id
    LEFT  JOIN customer_region  cr  ON ca.customer_id = cr.customer_id AND cr.rn = 1
),

-- Step 5: derive recency and average order value
customer_rfm_base AS (
    SELECT
        cj.*,
        -- recency: days from last purchase to dataset ceiling
        DATEDIFF(DAY, cj.last_purchase_date, dm.max_order_date) AS recency_days,
        -- avg_order_value: NULL when total_orders = 0 (honest, not 0)
        CASE
            WHEN cj.total_orders > 0
            THEN cj.total_revenue / cj.total_orders
        END AS avg_order_value
    FROM customer_joined cj
    CROSS JOIN dataset_max dm
),

-- Step 6: apply NTILE quintiles over the true customer-grain dataset
-- r_score: ASC order → smallest recency_days (most recent) → highest bucket
-- f_score: ASC order → highest total_orders → highest bucket
-- m_score: ASC order → highest total_revenue → highest bucket
customer_scored AS (
    SELECT
        *,
        NTILE(5) OVER (ORDER BY recency_days DESC)  AS r_score,
        NTILE(5) OVER (ORDER BY total_orders ASC)  AS f_score,
        NTILE(5) OVER (ORDER BY total_revenue ASC) AS m_score
    FROM customer_rfm_base
)

SELECT
    customer_id,
    region,
    loyalty_tier,
    signup_date,
    first_purchase_date,
    last_purchase_date,
    total_orders,
    CAST(total_revenue   AS DECIMAL(18,2))  AS total_revenue,
    total_units,
    CAST(avg_order_value AS DECIMAL(18,2))  AS avg_order_value,
    recency_days,
    total_orders                            AS frequency,
    CAST(total_revenue   AS DECIMAL(18,2))  AS monetary,
    r_score,
    f_score,
    m_score,
    CAST(r_score AS VARCHAR(1))
        + CAST(f_score AS VARCHAR(1))
        + CAST(m_score AS VARCHAR(1))       AS rfm_segment
INTO retail.customer_metrics
FROM customer_scored;
GO

In [ ]:
USE RetailDB;
GO

-- Quick sanity check: top 5 customers by total_revenue
SELECT TOP 5
    customer_id,
    loyalty_tier,
    total_orders,
    total_revenue,
    recency_days,
    r_score,
    f_score,
    m_score,
    rfm_segment
FROM retail.customer_metrics
ORDER BY total_revenue DESC;
GO

### Object 1b — `retail.customer_clv` (NEW)

**Grain:** One row per `customer_id` (known customers only, sourced from `customer_metrics`).

**Purpose:** Approximates Customer Lifetime Value in SQL using annualised purchase frequency × average order value × assumed 30% margin. Also assigns a quartile-based CLV segment (`Champions` / `Loyal` / `Promising` / `At Risk`).

**Business use:** CLV ranking, segment distribution, prioritisation for retention campaigns.

In [ ]:
USE RetailDB;
GO

IF OBJECT_ID('retail.customer_clv', 'U') IS NOT NULL
    DROP TABLE retail.customer_clv;
GO

WITH dataset_bounds AS (
    SELECT
        MIN(order_date) AS dataset_start,
        MAX(order_date) AS dataset_end,
        -- Observation window in days (used to annualise)
        DATEDIFF(DAY, MIN(order_date), MAX(order_date)) AS observation_days
    FROM retail.sales_enriched
),

customer_base AS (
    SELECT
        cm.customer_id,
        cm.loyalty_tier,
        cm.region,
        cm.total_orders,
        cm.total_revenue,
        cm.recency_days,
        cm.avg_order_value,
        cm.first_purchase_date,
        cm.last_purchase_date,
        db.observation_days,
        -- Active tenure in days (time between first and last purchase)
        DATEDIFF(DAY, cm.first_purchase_date, cm.last_purchase_date) AS tenure_days
    FROM retail.customer_metrics cm
    CROSS JOIN dataset_bounds db
),

clv_calculated AS (
    SELECT
        customer_id,
        loyalty_tier,
        region,
        total_orders,
        total_revenue                                                       AS historical_revenue,
        avg_order_value,
        recency_days,
        tenure_days,
        observation_days,

        -- Purchase rate: orders per day over active tenure (NULL if tenure=0, i.e. single purchase)
        CASE
            WHEN tenure_days > 0
            THEN CAST(total_orders AS FLOAT) / tenure_days
        END                                                                  AS purchase_rate_per_day,

        -- Annualised purchase frequency (365-day projection)
        CASE
            WHEN tenure_days > 0
            THEN CAST(total_orders AS FLOAT) / tenure_days * 365
            ELSE total_orders   -- single-purchase customers: use 1 order/year as floor
        END                                                                  AS predicted_annual_orders,

        -- Simple CLV: annualised orders × average order value × assumed margin (30%)
        -- Margin assumption is explicit and can be parameterised.
        CASE
            WHEN tenure_days > 0
            THEN CAST(total_orders AS FLOAT) / tenure_days * 365
                     * avg_order_value * 0.30
            ELSE avg_order_value * 0.30
        END                                                                  AS clv_1yr_estimate,

        -- CLV tier based on historical revenue quartiles
        -- NTILE(4) over revenue: 4=highest value customer
        NTILE(4) OVER (ORDER BY total_revenue ASC)                          AS clv_quartile

    FROM customer_base
),

clv_labelled AS (
    SELECT
        *,
        CASE clv_quartile
            WHEN 4 THEN 'Champions'
            WHEN 3 THEN 'Loyal'
            WHEN 2 THEN 'Promising'
            WHEN 1 THEN 'At Risk'
        END AS clv_segment
    FROM clv_calculated
)

SELECT
    customer_id,
    loyalty_tier,
    region,
    total_orders,
    CAST(historical_revenue      AS DECIMAL(18,2)) AS historical_revenue,
    CAST(avg_order_value         AS DECIMAL(18,2)) AS avg_order_value,
    recency_days,
    tenure_days,
    CAST(purchase_rate_per_day   AS DECIMAL(10,6)) AS purchase_rate_per_day,
    CAST(predicted_annual_orders AS DECIMAL(10,2)) AS predicted_annual_orders,
    CAST(clv_1yr_estimate        AS DECIMAL(18,2)) AS clv_1yr_estimate,
    clv_quartile,
    clv_segment
INTO retail.customer_clv
FROM clv_labelled;
GO

In [ ]:
USE RetailDB;
GO

SELECT TOP 5
    customer_id, loyalty_tier, historical_revenue,
    clv_1yr_estimate, clv_segment
FROM retail.customer_clv
ORDER BY clv_1yr_estimate DESC;
GO

### Object 2 — `retail.product_metrics`

**Grain:** One row per `product_id` (known products only, `product_known = 1`).

**Purpose:** Aggregates all sales activity per product, computes a return rate, and ranks products both overall and within their category by total revenue.

**Key metrics:**
- `return_rate`: percentage of line items with `delivery_status = 'Returned'` — left as `NULL` when the product has no rows (honest), returned as `DECIMAL(5,2)` otherwise.
- `revenue_rank_overall`: `RANK()` across all products ordered by `total_revenue DESC`.
- `revenue_rank_in_category`: `RANK()` partitioned by `category`, ordered by `total_revenue DESC`.

**Business use:** Product ranking, category performance comparison, supplier return-rate analysis.

In [ ]:
USE RetailDB;
GO

IF OBJECT_ID('retail.product_metrics', 'U') IS NOT NULL
    DROP TABLE retail.product_metrics;
GO

WITH product_agg AS (
    SELECT
        se.product_id,
        -- Take any non-NULL value — these are stable per product_id
        MAX(se.product_name)                                          AS product_name,
        MAX(se.category)                                              AS category,
        MAX(se.supplier_code)                                         AS supplier_code,
        MAX(se.base_price)                                            AS base_price,
        COUNT(DISTINCT se.order_id)                                   AS total_orders,
        SUM(COALESCE(se.quantity, 0))                                 AS total_units,
        SUM(COALESCE(se.Sales, 0))                                    AS total_revenue,
        SUM(COALESCE(se.GrossSales, 0))                              AS total_gross_sales,
        SUM(COALESCE(se.DiscountAmount, 0))                          AS total_discount,
        -- avg_discount_pct: NULL is honest when no rows exist
        AVG(se.discount_pct)                                          AS avg_discount_pct,
        -- return_rate: proportion of line items that were returned
        -- NULL guard: NULLIF prevents division by zero
        CAST(
            SUM(CASE WHEN se.delivery_status = 'Returned' THEN 1 ELSE 0 END) * 100.0
            / NULLIF(COUNT(*), 0)
        AS DECIMAL(5,2))                                              AS return_rate
    FROM retail.sales_enriched se
    WHERE se.product_known = 1
    GROUP BY se.product_id
),

product_ranked AS (
    SELECT
        *,
        -- Global revenue rank: ties share the same rank (gaps left intentionally by RANK)
        RANK() OVER (ORDER BY total_revenue DESC)                             AS revenue_rank_overall,
        -- Category-scoped rank: partitioned by category, same ordering
        RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC)       AS revenue_rank_in_category
    FROM product_agg
)

SELECT
    product_id,
    product_name,
    category,
    supplier_code,
    CAST(base_price         AS DECIMAL(18,2)) AS base_price,
    total_orders,
    total_units,
    CAST(total_revenue      AS DECIMAL(18,2)) AS total_revenue,
    CAST(total_gross_sales  AS DECIMAL(18,2)) AS total_gross_sales,
    CAST(total_discount     AS DECIMAL(18,2)) AS total_discount,
    avg_discount_pct,
    return_rate,
    revenue_rank_overall,
    revenue_rank_in_category
INTO retail.product_metrics
FROM product_ranked;
GO

In [ ]:
USE RetailDB;
GO

SELECT TOP 5
    product_id,
    product_name,
    category,
    total_revenue,
    return_rate,
    revenue_rank_overall,
    revenue_rank_in_category
FROM retail.product_metrics
ORDER BY revenue_rank_overall;
GO

### Object 2b — `retail.supplier_metrics` (NEW)

**Grain:** One row per `supplier_code` (known products only).

**Purpose:** Aggregates revenue, discount, return rate, and price-anomaly rate per supplier, then ranks suppliers by revenue and flags those whose return rate exceeds 2× the supplier average.

**Business use:** Supplier scorecard, return-rate outlier detection, negotiation input.

In [ ]:
USE RetailDB;
GO

IF OBJECT_ID('retail.supplier_metrics', 'U') IS NOT NULL
    DROP TABLE retail.supplier_metrics;
GO

WITH supplier_agg AS (
    SELECT
        se.supplier_code,
        COUNT(DISTINCT se.product_id)                           AS distinct_products,
        COUNT(DISTINCT se.order_id)                             AS total_orders,
        SUM(COALESCE(se.quantity, 0))                           AS total_units_sold,
        SUM(COALESCE(se.Sales, 0))                              AS total_revenue,
        SUM(COALESCE(se.DiscountAmount, 0))                     AS total_discount_given,
        AVG(se.discount_pct)                                    AS avg_discount_pct,
        -- Return rate at supplier level
        CAST(
            SUM(CASE WHEN se.delivery_status = 'Returned' THEN 1 ELSE 0 END) * 100.0
            / NULLIF(COUNT(*), 0)
        AS DECIMAL(5,2))                                        AS return_rate_pct,
        -- Price anomaly rate: what % of this supplier's line items have a price anomaly
        CAST(
            SUM(CASE WHEN se.PriceAnomaly = 1 THEN 1 ELSE 0 END) * 100.0
            / NULLIF(COUNT(*), 0)
        AS DECIMAL(5,2))                                        AS price_anomaly_rate_pct
    FROM retail.sales_enriched se
    WHERE se.product_known = 1      -- supplier is only meaningful for known products
    GROUP BY se.supplier_code
),

supplier_ranked AS (
    SELECT
        *,
        -- Revenue rank across all suppliers
        RANK() OVER (ORDER BY total_revenue DESC)               AS revenue_rank,
        -- Flag suppliers whose return rate is more than 2× the average
        AVG(return_rate_pct) OVER ()                            AS avg_return_rate_all,
        CASE
            WHEN return_rate_pct > 2 * AVG(return_rate_pct) OVER ()
            THEN 1 ELSE 0
        END                                                     AS high_return_flag
    FROM supplier_agg
)

SELECT
    supplier_code,
    distinct_products,
    total_orders,
    total_units_sold,
    CAST(total_revenue          AS DECIMAL(18,2)) AS total_revenue,
    CAST(total_discount_given   AS DECIMAL(18,2)) AS total_discount_given,
    avg_discount_pct,
    return_rate_pct,
    price_anomaly_rate_pct,
    revenue_rank,
    CAST(avg_return_rate_all    AS DECIMAL(5,2))  AS avg_return_rate_benchmark,
    high_return_flag
INTO retail.supplier_metrics
FROM supplier_ranked;
GO

In [ ]:
USE RetailDB;
GO

SELECT TOP 10 *
FROM retail.supplier_metrics
ORDER BY revenue_rank;
GO

### Object 3 — `retail.time_series_monthly`

**Grain:** One row per calendar year-month (e.g. `'2026-04'`, `'2026-05'`, `'2026-06'`).

**Purpose:** Provides a complete monthly revenue time series including a month-over-month (MoM) growth calculation. The first month will always have `NULL` for `prev_month_revenue` and `mom_growth_pct` because there is no prior month — this is expected and correct.

**Key metrics:**
- `prev_month_revenue`: `LAG(total_revenue, 1)` over months in chronological order.
- `mom_growth_pct`: `(total_revenue − prev_month_revenue) / prev_month_revenue × 100`, left as `NULL` for the first row.

**Business use:** Revenue trend charts, MoM growth analysis, seasonality patterns.

In [ ]:
USE RetailDB;
GO

IF OBJECT_ID('retail.time_series_monthly', 'U') IS NOT NULL
    DROP TABLE retail.time_series_monthly;
GO

WITH monthly_agg AS (
    SELECT
        -- FORMAT produces 'yyyy-MM' string for grouping and charting labels
        FORMAT(order_date, 'yyyy-MM')           AS year_month,
        YEAR(order_date)                        AS order_year,
        MONTH(order_date)                       AS order_month,
        COUNT(DISTINCT order_id)                AS total_orders,
        SUM(COALESCE(Sales, 0))                 AS total_revenue,
        SUM(COALESCE(GrossSales, 0))            AS total_gross_sales,
        SUM(COALESCE(DiscountAmount, 0))        AS total_discount,
        COUNT(DISTINCT customer_id)             AS unique_customers,
        COUNT(DISTINCT product_id)              AS unique_products
    FROM retail.sales_enriched
    GROUP BY
        FORMAT(order_date, 'yyyy-MM'),
        YEAR(order_date),
        MONTH(order_date)
),

monthly_with_aov AS (
    SELECT
        *,
        -- avg_order_value: NULL when no orders in month (honest)
        CASE
            WHEN total_orders > 0
            THEN total_revenue / total_orders
        END AS avg_order_value
    FROM monthly_agg
),

monthly_with_lag AS (
    SELECT
        *,
        -- LAG over chronological order gives previous month's revenue
        -- Ordered by year then month (INT) to avoid string sort issues
        LAG(total_revenue, 1) OVER (
            ORDER BY order_year, order_month
        ) AS prev_month_revenue
    FROM monthly_with_aov
)

SELECT
    CAST(year_month      AS VARCHAR(10))        AS year_month,
    order_year,
    order_month,
    total_orders,
    CAST(total_revenue      AS DECIMAL(18,2))   AS total_revenue,
    CAST(total_gross_sales  AS DECIMAL(18,2))   AS total_gross_sales,
    CAST(total_discount     AS DECIMAL(18,2))   AS total_discount,
    CAST(avg_order_value    AS DECIMAL(18,2))   AS avg_order_value,
    unique_customers,
    unique_products,
    CAST(prev_month_revenue AS DECIMAL(18,2))   AS prev_month_revenue,
    -- mom_growth_pct: NULL for first month; NULLIF guards against zero denominator
    CAST(
        (total_revenue - prev_month_revenue)
        / NULLIF(prev_month_revenue, 0) * 100
    AS DECIMAL(5,2))                            AS mom_growth_pct
INTO retail.time_series_monthly
FROM monthly_with_lag
ORDER BY order_year, order_month;
GO

In [ ]:
USE RetailDB;
GO

-- Expect 3 rows (Apr, May, Jun 2026); first row mom_growth_pct = NULL
SELECT *
FROM retail.time_series_monthly
ORDER BY order_year, order_month;
GO

### Object 4 — `retail.cohort_base`

**Grain:** One row per (`customer_id`, `order_id`) — all orders for known customers.

**Purpose:** Provides the raw cohort membership data that Python will pivot into a retention heatmap. Each row records:
- The customer's **acquisition month** (`customer_first_order_month`) — their cohort assignment.
- The **order month** — when this specific order occurred.
- The **cohort index** — how many months after acquisition this order was placed (0 = first month).

**Business use:** Cohort retention analysis — how many customers from each acquisition cohort return to purchase in months 1, 2, 3, etc.

In [ ]:
USE RetailDB;
GO

IF OBJECT_ID('retail.cohort_base', 'U') IS NOT NULL
    DROP TABLE retail.cohort_base;
GO

WITH customer_first_order AS (
    -- One row per customer: their earliest order date defines their cohort
    SELECT
        customer_id,
        MIN(order_date) AS first_order_date
    FROM retail.sales_enriched
    WHERE customer_known = 1
    GROUP BY customer_id
),

cohort_mapped AS (
    SELECT
        se.customer_id,
        se.order_id,
        se.order_date,
        -- Cohort label: the month of the customer's very first purchase
        FORMAT(cfo.first_order_date, 'yyyy-MM')         AS customer_first_order_month,
        -- Current order month label
        FORMAT(se.order_date, 'yyyy-MM')                AS order_month,
        -- Cohort index: integer months elapsed since first order
        -- Uses year and month arithmetic to avoid day-of-month edge cases
        (YEAR(se.order_date)  - YEAR(cfo.first_order_date))  * 12
        + (MONTH(se.order_date) - MONTH(cfo.first_order_date)) AS cohort_index
    FROM retail.sales_enriched se
    INNER JOIN customer_first_order cfo
        ON se.customer_id = cfo.customer_id
    WHERE se.customer_known = 1
)

SELECT
    customer_id,
    order_id,
    order_date,
    CAST(customer_first_order_month AS VARCHAR(10))  AS customer_first_order_month,
    CAST(order_month                AS VARCHAR(10))  AS order_month,
    cohort_index
INTO retail.cohort_base
FROM cohort_mapped;
GO

In [ ]:
USE RetailDB;
GO

SELECT TOP 5
    customer_id,
    order_id,
    order_date,
    customer_first_order_month,
    order_month,
    cohort_index
FROM retail.cohort_base
ORDER BY customer_id, order_date;
GO

### Object 4b — `retail.cohort_retention_rates` (NEW — pivoted retention %)

**Grain:** One row per (`cohort_month`, `cohort_index`).

**Purpose:** Converts raw `cohort_base` rows into a retention-rate matrix. `cohort_size` = distinct customers whose first purchase was in that cohort month. `active_customers` = distinct customers from that cohort who purchased at the given `cohort_index`. `retention_rate_pct` = `active_customers / cohort_size × 100`.

**Business use:** Cohort retention heatmap (Python/BI-ready — no further aggregation required).

In [ ]:
USE RetailDB;
GO

IF OBJECT_ID('retail.cohort_retention_rates', 'U') IS NOT NULL
    DROP TABLE retail.cohort_retention_rates;
GO

WITH cohort_sizes AS (
    -- Count of distinct customers who first purchased in each cohort month
    SELECT
        customer_first_order_month,
        COUNT(DISTINCT customer_id) AS cohort_size
    FROM retail.cohort_base
    WHERE cohort_index = 0          -- only acquisition month rows define cohort size
    GROUP BY customer_first_order_month
),

returning_customers AS (
    -- Count distinct customers active at each cohort_index
    SELECT
        cb.customer_first_order_month,
        cb.cohort_index,
        COUNT(DISTINCT cb.customer_id) AS active_customers
    FROM retail.cohort_base cb
    GROUP BY cb.customer_first_order_month, cb.cohort_index
)

SELECT
    rc.customer_first_order_month   AS cohort_month,
    cs.cohort_size,
    rc.cohort_index,
    rc.active_customers,
    -- Retention rate: what % of the original cohort is still purchasing at this index
    CAST(
        rc.active_customers * 100.0 / NULLIF(cs.cohort_size, 0)
    AS DECIMAL(5,2))                AS retention_rate_pct
INTO retail.cohort_retention_rates
FROM returning_customers rc
INNER JOIN cohort_sizes cs
    ON rc.customer_first_order_month = cs.customer_first_order_month
ORDER BY rc.customer_first_order_month, rc.cohort_index;
GO

In [ ]:
USE RetailDB;
GO

-- Verify: should show 100.00% at cohort_index=0 for all cohorts
SELECT *
FROM retail.cohort_retention_rates
ORDER BY cohort_month, cohort_index;
GO

### Object 5 — `retail.category_monthly_matrix`

**Grain:** One row per (`category`, `year_month`).

**Purpose:** Provides a pre-aggregated matrix of revenue and order count for every category × month combination. Python will pivot this into a heatmap — no further aggregation needed.

Rows where `product_known = 0` are excluded because `category` is `NULL` for unknown products and would create a meaningless `NULL` category bucket.

**Business use:** Category performance heatmap, identifying seasonal patterns within individual product categories.

In [ ]:
USE RetailDB;
GO

IF OBJECT_ID('retail.category_monthly_matrix', 'U') IS NOT NULL
    DROP TABLE retail.category_monthly_matrix;
GO

SELECT
    se.category,
    CAST(FORMAT(se.order_date, 'yyyy-MM') AS VARCHAR(10))  AS year_month,
    CAST(SUM(COALESCE(se.Sales, 0)) AS DECIMAL(18,2))      AS revenue,
    COUNT(DISTINCT se.order_id)                             AS orders
INTO retail.category_monthly_matrix
FROM retail.sales_enriched se
WHERE se.product_known = 1          -- excludes NULL category rows
GROUP BY
    se.category,
    FORMAT(se.order_date, 'yyyy-MM')
ORDER BY
    se.category,
    FORMAT(se.order_date, 'yyyy-MM');
GO

In [ ]:
USE RetailDB;
GO

SELECT TOP 5
    category,
    year_month,
    revenue,
    orders
FROM retail.category_monthly_matrix
ORDER BY category, year_month;
GO

### Object 6 — `retail.payment_risk_profile` (NEW)

**Grain:** One row per `payment_method`.

**Purpose:** Aggregates return rate, cancellation rate, price-anomaly rate, and average order value per payment method, then computes a weighted `composite_risk_score` (return × 0.5 + cancellation × 0.3 + anomaly × 0.2).

**Business use:** Payment-method risk dashboard, fraud/chargeback signal, checkout-friction analysis.

In [ ]:
USE RetailDB;
GO

IF OBJECT_ID('retail.payment_risk_profile', 'U') IS NOT NULL
    DROP TABLE retail.payment_risk_profile;
GO

WITH payment_agg AS (
    SELECT
        payment_method,
        COUNT(DISTINCT order_id)                                AS total_orders,
        SUM(COALESCE(Sales, 0))                                 AS total_revenue,
        -- Return rate by payment method
        CAST(
            SUM(CASE WHEN delivery_status = 'Returned' THEN 1 ELSE 0 END) * 100.0
            / NULLIF(COUNT(*), 0)
        AS DECIMAL(5,2))                                        AS return_rate_pct,
        -- Cancellation rate
        CAST(
            SUM(CASE WHEN delivery_status = 'Cancelled' THEN 1 ELSE 0 END) * 100.0
            / NULLIF(COUNT(*), 0)
        AS DECIMAL(5,2))                                        AS cancellation_rate_pct,
        -- Price anomaly rate — 'Unknown' payment methods may correlate with data issues
        CAST(
            SUM(CASE WHEN PriceAnomaly = 1 THEN 1 ELSE 0 END) * 100.0
            / NULLIF(COUNT(*), 0)
        AS DECIMAL(5,2))                                        AS price_anomaly_rate_pct,
        AVG(CAST(Sales AS FLOAT))                               AS avg_order_value
    FROM retail.sales_enriched
    GROUP BY payment_method
),

payment_ranked AS (
    SELECT
        *,
        -- Risk score: composite of return + cancellation + anomaly rates
        -- Simple additive; weights can be tuned per business rules
        CAST(
            return_rate_pct * 0.5
            + cancellation_rate_pct * 0.3
            + price_anomaly_rate_pct * 0.2
        AS DECIMAL(5,2))                                        AS composite_risk_score,
        RANK() OVER (ORDER BY total_revenue DESC)               AS revenue_rank
    FROM payment_agg
)

SELECT
    payment_method,
    total_orders,
    CAST(total_revenue      AS DECIMAL(18,2)) AS total_revenue,
    return_rate_pct,
    cancellation_rate_pct,
    price_anomaly_rate_pct,
    CAST(avg_order_value    AS DECIMAL(18,2)) AS avg_order_value,
    composite_risk_score,
    revenue_rank
INTO retail.payment_risk_profile
FROM payment_ranked;
GO

In [ ]:
USE RetailDB;
GO

SELECT * FROM retail.payment_risk_profile ORDER BY composite_risk_score DESC;
GO

### Object 7 — `retail.mba_order_product_matrix` (NEW — pre-aggregation for Python Apriori)

**Grain:** One row per (`order_id`, `product_id`), fulfilled orders only.

**Purpose:** Pre-aggregates the order × product matrix in SQL so the Python MBA step (Apriori) runs in O(orders) instead of O(line_items × products). Returned/Cancelled orders are excluded because they do not represent market-basket behaviour.

**Business use:** Input to `mlxtend.apriori` in the downstream Python EDA notebook. Python writes the resulting `retail.mba_association_rules` table back to SQL for BI consumption.

In [ ]:
USE RetailDB;
GO

IF OBJECT_ID('retail.mba_order_product_matrix', 'U') IS NOT NULL
    DROP TABLE retail.mba_order_product_matrix;
GO

SELECT
    order_id,
    product_id,
    product_name,
    category,
    SUM(quantity) AS total_quantity    -- aggregate if same product appears on multiple lines
INTO retail.mba_order_product_matrix
FROM retail.sales_enriched
WHERE product_known = 1
  AND delivery_status NOT IN ('Returned', 'Cancelled')   -- only fulfilled orders
GROUP BY order_id, product_id, product_name, category;
GO

### Gold Layer Indexes (NEW — for Power BI DirectQuery)

Non-clustered indexes on the columns Power BI filters and sorts by most often. Keeps DirectQuery responsive at current scale without needing Import Mode.

In [ ]:
USE RetailDB;
GO

-- customer_metrics: Power BI filters most often by loyalty_tier and region
CREATE NONCLUSTERED INDEX IX_customer_metrics_tier_region
    ON retail.customer_metrics (loyalty_tier, region)
    INCLUDE (total_revenue, rfm_segment, r_score, f_score, m_score);

-- product_metrics: filtered by category and ranked by revenue
CREATE NONCLUSTERED INDEX IX_product_metrics_category_revenue
    ON retail.product_metrics (category, revenue_rank_in_category)
    INCLUDE (total_revenue, return_rate, product_name);

-- time_series_monthly: always filtered/sorted by date
CREATE NONCLUSTERED INDEX IX_tsm_year_month
    ON retail.time_series_monthly (order_year, order_month)
    INCLUDE (total_revenue, mom_growth_pct, unique_customers);

-- cohort_retention_rates: heatmap queries by cohort_month and cohort_index
CREATE NONCLUSTERED INDEX IX_cohort_month_index
    ON retail.cohort_retention_rates (cohort_month, cohort_index)
    INCLUDE (retention_rate_pct, cohort_size);
GO

### Handoff to Python

The Analytics Layer is complete. The 10 tables below are the **only inputs** consumed by the downstream Python EDA notebook — no additional SQL computation is required on the Python side.

| Table | Python use |
|---|---|
| `retail.customer_metrics` | RFM scatter plots, loyalty tier distributions |
| `retail.customer_clv` | CLV ranking, segment distribution |
| `retail.product_metrics` | Revenue rank bars, return rate analysis |
| `retail.supplier_metrics` | Supplier scorecard, return-rate outliers |
| `retail.time_series_monthly` | Revenue line chart, MoM growth bars |
| `retail.cohort_base` | Raw cohort retention pivot |
| `retail.cohort_retention_rates` | Pre-pivoted retention heatmap |
| `retail.category_monthly_matrix` | Category × month heatmap |
| `retail.payment_risk_profile` | Payment method risk dashboard |
| `retail.mba_order_product_matrix` | Apriori market basket (Python only) |

Python will connect to **RetailDB**, load each table with `pd.read_sql()`, and proceed directly to visualisation.

In [ ]:
USE RetailDB;
GO

-- ============================================================
-- Full Gold Validation Sweep
-- Covers every Gold-layer table including the new ones.
-- Returns (check_name, value, expected_value, status).
-- status ∈ { PASS, FAIL, INFO }
-- ============================================================

SELECT check_name, CAST(value AS NVARCHAR(50)) AS value, expected_value, status
FROM (

    SELECT 'customer_metrics: total rows'               AS check_name,
           COUNT(*)                                      AS value,
           'N/A'                                         AS expected_value,
           'INFO'                                        AS status
    FROM retail.customer_metrics

    UNION ALL SELECT 'customer_metrics: NULL rfm_segment',
           COUNT(*), '0', CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
    FROM retail.customer_metrics WHERE rfm_segment IS NULL

    UNION ALL SELECT 'customer_clv: total rows',
           COUNT(*), 'N/A', 'INFO'
    FROM retail.customer_clv

    UNION ALL SELECT 'customer_clv: NULL clv_1yr_estimate',
           COUNT(*), '0', CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
    FROM retail.customer_clv WHERE clv_1yr_estimate IS NULL

    UNION ALL SELECT 'product_metrics: total rows',
           COUNT(*), 'N/A', 'INFO'
    FROM retail.product_metrics

    UNION ALL SELECT 'product_metrics: NULL category',
           COUNT(*), '0', CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
    FROM retail.product_metrics WHERE category IS NULL

    UNION ALL SELECT 'supplier_metrics: high_return_flag count',
           SUM(high_return_flag), 'N/A', 'INFO'
    FROM retail.supplier_metrics

    UNION ALL SELECT 'time_series_monthly: total months',
           COUNT(*), 'N/A', 'INFO'
    FROM retail.time_series_monthly

    UNION ALL SELECT 'time_series_monthly: NULL mom_growth_pct (expect 1)',
           COUNT(*), '1', CASE WHEN COUNT(*) = 1 THEN 'PASS' ELSE 'FAIL' END
    FROM retail.time_series_monthly WHERE mom_growth_pct IS NULL

    UNION ALL SELECT 'cohort_retention_rates: cohort_index=0 all 100%',
           SUM(CASE WHEN cohort_index = 0 AND retention_rate_pct <> 100 THEN 1 ELSE 0 END),
           '0',
           CASE WHEN SUM(CASE WHEN cohort_index = 0
                              AND retention_rate_pct <> 100 THEN 1 ELSE 0 END) = 0
                THEN 'PASS' ELSE 'FAIL' END
    FROM retail.cohort_retention_rates

    UNION ALL SELECT 'cohort_base: total rows',
           COUNT(*), 'N/A', 'INFO'
    FROM retail.cohort_base

    UNION ALL SELECT 'category_monthly_matrix: total rows',
           COUNT(*), 'N/A', 'INFO'
    FROM retail.category_monthly_matrix

    UNION ALL SELECT 'payment_risk_profile: total payment methods',
           COUNT(*), 'N/A', 'INFO'
    FROM retail.payment_risk_profile

    UNION ALL SELECT 'mba_order_product_matrix: total rows',
           COUNT(*), 'N/A', 'INFO'
    FROM retail.mba_order_product_matrix

) AS validation
ORDER BY
    CASE status WHEN 'FAIL' THEN 0 WHEN 'PASS' THEN 1 ELSE 2 END,
    check_name;
GO

In [ ]:
USE RetailDB;
GO

SELECT 
    (SELECT COUNT(*) FROM retail.customers_clean)                    AS clean_customers,
    (SELECT COUNT(*) FROM retail.customer_metrics)                    AS metrics_customers,
    (SELECT COUNT(DISTINCT customer_id) FROM retail.sales_enriched 
        WHERE customer_known = 1)                                     AS customers_with_sales,
    (SELECT COUNT(DISTINCT customer_id) FROM retail.sales_enriched)   AS all_sales_customers;
GO

## 8. EDA Queries

Quick exploratory queries against the enriched fact table. These are the first queries a downstream analyst would run.


In [ ]:
USE RetailDB;
GO

-- Top 10 products by revenue
SELECT TOP 10
    product_name,
    category,
    SUM(Sales)          AS total_sales,
    SUM(quantity)       AS units_sold
FROM retail.sales_enriched
GROUP BY product_name, category
ORDER BY total_sales DESC;
GO

-- Revenue by category
SELECT
    category,
    SUM(Sales)                                AS total_sales,
    CAST(SUM(Sales) * 100.0 / SUM(SUM(Sales)) OVER () AS DECIMAL(5,2)) AS pct_of_total
FROM retail.sales_enriched
GROUP BY category
ORDER BY total_sales DESC;
GO

-- Monthly sales trend
SELECT
    FORMAT(order_date, 'yyyy-MM') AS month,
    COUNT(DISTINCT order_id)      AS orders,
    SUM(Sales)                    AS total_sales
FROM retail.sales_enriched
GROUP BY FORMAT(order_date, 'yyyy-MM')
ORDER BY month;
GO

## 9. Final Summary

This pipeline transformed raw, messy CSV data into a clean, analysis-ready schema. The following tables are now available for EDA and reporting:

| Table | Purpose |
|---|---|
| `retail.customers_clean` | Deduplicated, standardized customer master |
| `retail.products_clean` | Deduplicated catalog with imputed prices |
| `retail.sales_clean` | Filtered, standardized transaction records |
| `retail.sales_enriched` | Denormalized fact table with computed metrics |
| `retail.order_summary` | Order-level aggregations |
| `retail.customer_metrics` | RFM scores and customer grain metrics |
| `retail.customer_clv` | Customer lifetime value estimates |
| `retail.product_metrics` | Product revenue rank and return rates |
| `retail.supplier_metrics` | Supplier scorecard |
| `retail.time_series_monthly` | Monthly revenue and MoM growth |
| `retail.cohort_base` / `cohort_retention_rates` | Cohort retention analysis |
| `retail.category_monthly_matrix` | Category × month revenue matrix |
| `retail.payment_risk_profile` | Payment method risk scores |
| `retail.mba_order_product_matrix` | Pre-agg for market-basket (Apriori) |

### Next Steps
1. **EDA Queries** — top products, top customers, monthly trends, discount impact (see section 8 above)
2. **Visualization** — connect to Power BI or Python for charts
3. **Modeling** — customer segmentation, churn prediction, demand forecasting
